# 02. Feature engineering: churn driver diagnostic

**Purpose.** Turn the explorer's findings into model inputs, one feature at a time. Each feature has a business meaning, a formula, the exact code, the finding that motivates it, the direction we expect, a before/after example row, a check against churn on the training rows only, and a keep-or-drop verdict.

**What this notebook writes:** `reports/modeling/features.parquet` (all 7,043 rows, every feature, the target, and a `split` column), `reports/modeling/features.md` (the feature table) and `reports/modeling/feature_list.json` (which features the models use).

**What it cannot do.** The checks show association with churn, not cause. A feature that helps prediction is not a lever until it is tested as one.

## 1. Setup

Plain words: load the libraries, fix the random seed (a number that makes every random step repeat exactly) and print library versions so the run can be reproduced. All feature code lives in `src/features.py` so the production pipeline uses the same definitions.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
PROJECT = os.path.abspath('..')
sys.path.insert(0, os.path.join(PROJECT, 'src'))
import numpy as np, pandas as pd, sklearn, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import features as F
import model_utils as M
SEED = F.SEED
np.random.seed(SEED)
os.makedirs(f'{PROJECT}/reports/modeling/charts', exist_ok=True)
print('seed', SEED, '| python', sys.version.split()[0], '| pandas', pd.__version__, '| numpy', np.__version__,
      '| sklearn', sklearn.__version__, '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)

seed 42 | python 3.13.12 | pandas 2.3.3 | numpy 2.4.6 | sklearn 1.9.1 | scipy 1.18.1 | matplotlib 3.11.2


The output lists the seed (42) and the library versions. If a later run shows different versions, small numeric differences are possible.

## 2. Data and the train / held-out split

**Hold out 20% of customers before any feature is looked at**

1. **Why this method.** Anything we look at to choose features can leak the answer into the model. Setting aside a test set first, and never using it here, keeps the final score honest. The explorer's handoff asks for a stratified random 20% split with seed 42. Rows are one customer each, so rows are independent and a random split is valid. There is no date, so a time split is impossible.
2. **Alternatives considered.**
   - Plain random split: would let the leaver share drift (for example 25% in one part, 28% in the other) by chance. Not used.
   - Group split by customer: no customer appears twice, so it would equal a random split. Not needed.
   - Out-of-time split: needs a date column, which this data does not have.
3. **How it works.** Stratified sampling draws 20% of the leavers and 20% of the stayers separately, so both parts keep the 26.5% churn rate. Test size = 0.20 x 7,043 = 1,409 customers. Decision rule, fixed before the result: the split is made once with seed 42, the test rows are touched exactly once in notebook 03, and every check in this notebook uses training rows only.
4. **How to read the output.** Check that the two parts show the same churn rate (about 26.5%) and that test has about 374 leavers. If the rates differed by more than a point, the split would be re-examined.

In [3]:
df = pd.read_parquet(f'{PROJECT}/reports/exploration/clean.parquet')
df['split'] = F.split_data(df)
summ = df.groupby('split')[F.TARGET].agg(customers='count', leavers='sum', churn_rate='mean')
summ['churn_rate_pct'] = (100*summ['churn_rate']).round(2)
display(summ.drop(columns='churn_rate'))
is_train = df['split'].eq('train')
print('Training rows used for every check below:', is_train.sum())

,customers,leavers,churn_rate_pct
split,,,
test,1409,374,26.54
train,5634,1495,26.54


Training rows used for every check below: 5634


The training part holds 5,634 customers and the held-out test part 1,409. Churn rates match within a few hundredths of a point, so the split is balanced. From here on, every statistic is computed on training rows only.

## 3. Columns left out, and a leakage check

**Check the billing-total columns the explorer asked us to leave out**

1. **Why this method.** TotalCharges, avg_monthly_spend and price_vs_history_pct are built from lifetime billing. For leavers, billing stops at exit, so these columns may carry the outcome itself. The handoff says to add them back only after a leakage check. The data cannot prove when they were measured, so this check can only show how much they would tell us on their own.
2. **Alternatives considered.**
   - Ask the data owner when the totals were measured: the right answer, but not available in this project.
   - Add them and compare cross-validated scores: a big jump would be a warning sign, not proof of either answer, so it cannot clear them. Used only as a warning light.
   - Drop silently: loses the chance to show the manager what was given up.
3. **How it works.** Single-column AUC (area under the ROC curve: the chance the column ranks a random leaver above a random stayer; 0.5 is a coin flip, 1.0 is perfect) for each column, on training rows. Formula: $AUC=P(s_i>s_j\mid y_i=1,y_j=0)$, where $s$ is the column value read as a risk score. We use $\max(AUC,1-AUC)$ so direction does not matter. Decision rule, fixed before the result: these columns stay out of the model unless the data owner confirms they are measured before the churn event. An AUC far above that of tenure and MonthlyCharges would be taken as a leakage warning.
4. **How to read the output.** Compare the billing columns with tenure and MonthlyCharges. Similar values mean the billing columns add little and the exclusion costs little. A much larger value would be a warning that they carry the outcome.

In [5]:
from sklearn.metrics import roc_auc_score
trn = df[is_train]
rows = []
for c in ['tenure','MonthlyCharges','TotalCharges','avg_monthly_spend','price_vs_history_pct']:
    x = trn[c].fillna(trn[c].median())
    a = roc_auc_score(trn[F.TARGET], x)
    rows.append(dict(column=c, single_column_auc=round(max(a,1-a),3), used_in_model=c in ('tenure','MonthlyCharges')))
display(pd.DataFrame(rows))

,column,single_column_auc,used_in_model
0,tenure,0.735,True
1,MonthlyCharges,0.624,True
2,TotalCharges,0.647,False
3,avg_monthly_spend,0.624,False
4,price_vs_history_pct,0.504,False


Read the table against the rule. On training rows the billing columns reach AUC 0.647 (TotalCharges), 0.624 (avg_monthly_spend) and 0.504 (price_vs_history_pct), none above tenure at 0.735, so there is no sign of an outcome-carrying column and leaving them out gives up little ranking power. They stay excluded because the rule asks for the data owner's confirmation, which we do not have. (`TotalCharges` also has 11 blank values, all tenure-0 accounts.) This is a precaution, not proof that they leak.

## 4. Engineered features, one at a time

Each feature below follows the same steps: (a) a specification cell, (b) a code cell that builds it from the raw columns, shows one example row before and after, and checks it against churn on the training rows, (c) a note on what the output shows.

**Common method for the check.** Why: a feature that does not move churn adds noise and a place to overfit. Alternatives: a full model with and without the feature (done in section 5 for the whole set; too slow to repeat per feature), or mutual information (same story as the simple rates here, harder to read). How: a binary feature is compared by churn rate when 1 versus when 0; a count or number is compared by Spearman rank correlation (a number from -1 to 1 saying how steadily churn rises or falls as the feature rises; 0 means no steady trend) and by churn rate per value or per decile bin. Decision rule (fixed before any result): **binary feature KEEP if the gap is at least 5 points and each group has at least 100 training rows; other features KEEP if |Spearman| is at least 0.10.** How to read: the chart shows churn rate by value against the training base rate (the red dashed line).

**Scales for reading the numbers (conventions, not laws).** Spearman rho: under 0.10 negligible, 0.10 to 0.30 small, 0.30 to 0.50 medium, over 0.50 large. Churn-rate gap, in percentage points against the 26.5% base rate: under 5 negligible, 5 to 15 small, 15 to 30 medium, over 30 large. **Why these thresholds.** |rho| of 0.10 is the usual boundary between noise-level and small association; with 5,634 rows, a rho of 0.03 is already statistically detectable but useless for ranking, so a practical floor is needed. A 5-point gap means 5 extra leavers per 100 customers, about the smallest difference a retention action could act on; the 100-row floor per group comes from the handoff (rates on fewer rows are too noisy). They are screening rules, not tests: a feature near a threshold with a small group should be read with its interval. **Interval method.** Each note below adds an approximate 95% interval from closed formulas: for a gap, $\Delta\pm1.96\sqrt{\hat p_1(1-\hat p_1)/n_1+\hat p_0(1-\hat p_0)/n_0}$ (normal approximation to a difference of two proportions); for rho, Fisher's transform $\tanh(\operatorname{atanh}(\rho)\pm1.96/\sqrt{n-3})$. A bootstrap (resample customers 1,000 times) would give nearly the same width at this sample size; the formulas are used because they can be checked by hand. The intervals are computed from the same training rows when the notebook is built and are not printed by the code cells.

In [6]:
# Build all features one by one below; `df` holds every row, `feat` collects the checks.
feat = {}
base_rate = df.loc[is_train, F.TARGET].mean()
print('Training base churn rate:', round(100*base_rate, 2), '%')

Training base churn rate: 26.54 %


The base rate on training rows is the line each feature's bars are compared to.

### 4.1 `contract_ord`
- **Business meaning:** Contract length as an ordered number: 0 month-to-month, 1 one year, 2 two years.
- **Formula:** $$\text{contract\_ord}=\begin{cases}0&\text{month-to-month}\\1&\text{one year}\\2&\text{two year}\end{cases}$$
- **Exact code expression:** `df['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})`
- **Input columns:** `Contract`
- **Why it exists (explorer finding or hypothesis):** H1 / finding 1: churn is 42.7%, 11.3%, 2.8% by contract. The order is monotone, so one ordered number lets a linear model use the order with one coefficient.
- **Expected direction of its link to churn:** negative (longer contract, less churn)
- **Example row (row index 1), before and after:** inputs Contract='One year' give `contract_ord` = 1.
- **Type:** numeric (0/1/2). Built row by row, nothing is fitted, so it cannot leak across the split.

In [7]:
# 4.1 contract_ord
EX = 1
print('before (inputs):', df.loc[EX, ['Contract']].to_dict())
df['contract_ord'] = (df['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})).astype(float)
print('after: contract_ord =', df.loc[EX, 'contract_ord'])
res = F.check_feature(df.loc[is_train, 'contract_ord'], df.loc[is_train, F.TARGET])
feat['contract_ord'] = res
F.plot_feature_check('contract_ord', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_contract_ord.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'Contract': 'One year'}
after: contract_ord = 1.0
Spearman rho with churn -0.41 (approx. 95% interval -0.43 to -0.39); churn rate from 42.7% (lowest value/bin) to 2.9% (highest)
rule: |Spearman rho| >= 0.10 -> KEEP


**What the output shows.** On the 5,634 training customers: Spearman rho with churn -0.41 (approx. 95% interval -0.43 to -0.39); churn rate from 42.7% (lowest value/bin) to 2.9% (highest). Rule (|Spearman rho| >= 0.10) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho -0.41 is medium (approximate 95% interval -0.43 to -0.39). Final verdicts are set in section 5 after the redundancy check.

### 4.2 `m2m_fiber`
- **Business meaning:** Month-to-month contract and fiber-optic internet together.
- **Formula:** $$\text{m2m\_fiber}=\mathbb{1}[\text{Contract}=\text{Month-to-month}]\cdot\mathbb{1}[\text{InternetService}=\text{Fiber optic}]$$
- **Exact code expression:** `((df['Contract'] == 'Month-to-month') & (df['InternetService'] == 'Fiber optic'))`
- **Input columns:** `Contract`, `InternetService`
- **Why it exists (explorer finding or hypothesis):** H2, H9 / finding 3: fiber on month-to-month leaves at 55% against 32% for DSL. A linear model adds effects; this flag lets it see the joint group.
- **Expected direction of its link to churn:** positive
- **Example row (row index 4), before and after:** inputs Contract='Month-to-month', InternetService='Fiber optic' give `m2m_fiber` = 1.
- **Type:** binary. Built row by row, nothing is fitted, so it cannot leak across the split.

In [8]:
# 4.2 m2m_fiber
EX = 4
print('before (inputs):', df.loc[EX, ['Contract', 'InternetService']].to_dict())
df['m2m_fiber'] = (((df['Contract'] == 'Month-to-month') & (df['InternetService'] == 'Fiber optic'))).astype(float)
print('after: m2m_fiber =', df.loc[EX, 'm2m_fiber'])
res = F.check_feature(df.loc[is_train, 'm2m_fiber'], df.loc[is_train, F.TARGET])
feat['m2m_fiber'] = res
F.plot_feature_check('m2m_fiber', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_m2m_fiber.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'Contract': 'Month-to-month', 'InternetService': 'Fiber optic'}
after: m2m_fiber = 1.0
churn 55.1% when 1 (n=1,707) vs 14.1% when 0 (n=3,927); gap +40.9 points (approx. 95% interval +38.3 to +43.5); Spearman rho +0.43 (approx. 95% interval +0.40 to +0.45)
rule: gap >= 5 points and both groups >= 100 rows -> KEEP


**What the output shows.** On the 5,634 training customers: churn 55.1% when 1 (n=1,707) vs 14.1% when 0 (n=3,927); gap +40.9 points (approx. 95% interval +38.3 to +43.5); Spearman rho +0.43 (approx. 95% interval +0.40 to +0.45). Rule (gap >= 5 points and both groups >= 100 rows) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho +0.43 is medium (approximate 95% interval +0.40 to +0.45); the gap of +40.9 points is large (approximate 95% interval +38.3 to +43.5). Final verdicts are set in section 5 after the redundancy check.

### 4.3 `m2m_echeck`
- **Business meaning:** Month-to-month contract and electronic-check payment together.
- **Formula:** $$\text{m2m\_echeck}=\mathbb{1}[\text{Contract}=\text{Month-to-month}]\cdot\mathbb{1}[\text{PaymentMethod}=\text{Electronic check}]$$
- **Exact code expression:** `((df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] == 'Electronic check'))`
- **Input columns:** `Contract`, `PaymentMethod`
- **Why it exists (explorer finding or hypothesis):** H4 / finding 4: electronic-check payers on month-to-month leave at 54%, against about 33% for automatic payers on the same contract.
- **Expected direction of its link to churn:** positive
- **Example row (row index 0), before and after:** inputs Contract='Month-to-month', PaymentMethod='Electronic check' give `m2m_echeck` = 1.
- **Type:** binary. Built row by row, nothing is fitted, so it cannot leak across the split.

In [7]:
# 4.3 m2m_echeck
EX = 0
print('before (inputs):', df.loc[EX, ['Contract', 'PaymentMethod']].to_dict())
df['m2m_echeck'] = (((df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] == 'Electronic check'))).astype(float)
print('after: m2m_echeck =', df.loc[EX, 'm2m_echeck'])
res = F.check_feature(df.loc[is_train, 'm2m_echeck'], df.loc[is_train, F.TARGET])
feat['m2m_echeck'] = res
F.plot_feature_check('m2m_echeck', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_m2m_echeck.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'Contract': 'Month-to-month', 'PaymentMethod': 'Electronic check'}
after: m2m_echeck = 1.0
churn 54.2% when 1 (n=1,484) vs 16.6% when 0 (n=4,150); gap +37.6 points (approx. 95% interval +34.8 to +40.4); Spearman rho +0.38 (approx. 95% interval +0.35 to +0.40)
rule: gap >= 5 points and both groups >= 100 rows -> KEEP


**What the output shows.** On the 5,634 training customers: churn 54.2% when 1 (n=1,484) vs 16.6% when 0 (n=4,150); gap +37.6 points (approx. 95% interval +34.8 to +40.4); Spearman rho +0.38 (approx. 95% interval +0.35 to +0.40). Rule (gap >= 5 points and both groups >= 100 rows) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho +0.38 is medium (approximate 95% interval +0.35 to +0.40); the gap of +37.6 points is large (approximate 95% interval +34.8 to +40.4). Final verdicts are set in section 5 after the redundancy check.

### 4.4 `m2m_first_year`
- **Business meaning:** Month-to-month contract and 12 months or less with the company.
- **Formula:** $$\text{m2m\_first\_year}=\mathbb{1}[\text{Contract}=\text{Month-to-month}]\cdot\mathbb{1}[\text{tenure}\le 12]$$
- **Exact code expression:** `((df['Contract'] == 'Month-to-month') & (df['tenure'] <= 12))`
- **Input columns:** `Contract`, `tenure`
- **Why it exists (explorer finding or hypothesis):** H1, H3 / finding 5: month-to-month customers in year one are 28% of the base and hold 55% of leavers.
- **Expected direction of its link to churn:** positive
- **Example row (row index 0), before and after:** inputs Contract='Month-to-month', tenure=np.int64(1) give `m2m_first_year` = 1.
- **Type:** binary. Built row by row, nothing is fitted, so it cannot leak across the split.

In [8]:
# 4.4 m2m_first_year
EX = 0
print('before (inputs):', df.loc[EX, ['Contract', 'tenure']].to_dict())
df['m2m_first_year'] = (((df['Contract'] == 'Month-to-month') & (df['tenure'] <= 12))).astype(float)
print('after: m2m_first_year =', df.loc[EX, 'm2m_first_year'])
res = F.check_feature(df.loc[is_train, 'm2m_first_year'], df.loc[is_train, F.TARGET])
feat['m2m_first_year'] = res
F.plot_feature_check('m2m_first_year', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_m2m_first_year.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'Contract': 'Month-to-month', 'tenure': 1}
after: m2m_first_year = 1.0
churn 51.1% when 1 (n=1,594) vs 16.9% when 0 (n=4,040); gap +34.2 points (approx. 95% interval +31.5 to +36.9); Spearman rho +0.35 (approx. 95% interval +0.33 to +0.37)
rule: gap >= 5 points and both groups >= 100 rows -> KEEP


**What the output shows.** On the 5,634 training customers: churn 51.1% when 1 (n=1,594) vs 16.9% when 0 (n=4,040); gap +34.2 points (approx. 95% interval +31.5 to +36.9); Spearman rho +0.35 (approx. 95% interval +0.33 to +0.37). Rule (gap >= 5 points and both groups >= 100 rows) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho +0.35 is medium (approximate 95% interval +0.33 to +0.37); the gap of +34.2 points is large (approximate 95% interval +31.5 to +36.9). Final verdicts are set in section 5 after the redundancy check.

### 4.5 `first_year`
- **Business meaning:** Customer is in the first 12 months.
- **Formula:** $$\text{first\_year}=\mathbb{1}[\text{tenure}\le 12]$$
- **Exact code expression:** `(df['tenure'] <= 12)`
- **Input columns:** `tenure`
- **Why it exists (explorer finding or hypothesis):** H3 / finding 2: 47.4% churn in months 0-12 against 17.1% after. A step is easier for a linear model than a curve.
- **Expected direction of its link to churn:** positive
- **Example row (row index 0), before and after:** inputs tenure=np.int64(1) give `first_year` = 1.
- **Type:** binary. Built row by row, nothing is fitted, so it cannot leak across the split.

In [9]:
# 4.5 first_year
EX = 0
print('before (inputs):', df.loc[EX, ['tenure']].to_dict())
df['first_year'] = ((df['tenure'] <= 12)).astype(float)
print('after: first_year =', df.loc[EX, 'first_year'])
res = F.check_feature(df.loc[is_train, 'first_year'], df.loc[is_train, F.TARGET])
feat['first_year'] = res
F.plot_feature_check('first_year', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_first_year.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'tenure': 1}
after: first_year = 1.0


churn 47.3% when 1 (n=1,737) vs 17.3% when 0 (n=3,897); gap +30.1 points (approx. 95% interval +27.4 to +32.7); Spearman rho +0.31 (approx. 95% interval +0.29 to +0.34)
rule: gap >= 5 points and both groups >= 100 rows -> KEEP


**What the output shows.** On the 5,634 training customers: churn 47.3% when 1 (n=1,737) vs 17.3% when 0 (n=3,897); gap +30.1 points (approx. 95% interval +27.4 to +32.7); Spearman rho +0.31 (approx. 95% interval +0.29 to +0.34). Rule (gap >= 5 points and both groups >= 100 rows) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho +0.31 is medium (approximate 95% interval +0.29 to +0.34); the gap of +30.1 points is large (approximate 95% interval +27.4 to +32.7). Final verdicts are set in section 5 after the redundancy check.

### 4.6 `log_tenure`
- **Business meaning:** Log of tenure, so early months count more than late months.
- **Formula:** $$\text{log\_tenure}=\ln(1+\text{tenure})$$
- **Exact code expression:** `np.log1p(df['tenure'])`
- **Input columns:** `tenure`
- **Why it exists (explorer finding or hypothesis):** Finding 2 / chart 04: churn falls fast in the first months and flattens after. A log curve matches that shape.
- **Expected direction of its link to churn:** negative
- **Example row (row index 0), before and after:** inputs tenure=np.int64(1) give `log_tenure` = 0.693147.
- **Type:** numeric. Built row by row, nothing is fitted, so it cannot leak across the split.

In [10]:
# 4.6 log_tenure
EX = 0
print('before (inputs):', df.loc[EX, ['tenure']].to_dict())
df['log_tenure'] = (np.log1p(df['tenure'])).astype(float)
print('after: log_tenure =', df.loc[EX, 'log_tenure'])
res = F.check_feature(df.loc[is_train, 'log_tenure'], df.loc[is_train, F.TARGET])
feat['log_tenure'] = res
F.plot_feature_check('log_tenure', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_log_tenure.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'tenure': 1}
after: log_tenure = 0.6931471805599453
Spearman rho with churn -0.36 (approx. 95% interval -0.38 to -0.34); churn rate from 57.5% (lowest value/bin) to 3.8% (highest)
rule: |Spearman rho| >= 0.10 -> KEEP


**What the output shows.** On the 5,634 training customers: Spearman rho with churn -0.36 (approx. 95% interval -0.38 to -0.34); churn rate from 57.5% (lowest value/bin) to 3.8% (highest). Rule (|Spearman rho| >= 0.10) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho -0.36 is medium (approximate 95% interval -0.38 to -0.34). Final verdicts are set in section 5 after the redundancy check.

### 4.7 `internet_no_techsupport`
- **Business meaning:** Has internet but no tech-support add-on.
- **Formula:** $$\text{internet\_no\_techsupport}=\mathbb{1}[\text{InternetService}\ne\text{No}]\cdot\mathbb{1}[\text{TechSupport}=\text{No}]$$
- **Exact code expression:** `((df['InternetService'] != 'No') & (df['TechSupport'] == 'No'))`
- **Input columns:** `InternetService`, `TechSupport`
- **Why it exists (explorer finding or hypothesis):** H5 / finding 'also notable': internet customers without tech support leave at 42% against 15% with it. Customers with no internet are 'No' on tech support but leave rarely (7%), so the flag keeps them apart.
- **Expected direction of its link to churn:** positive
- **Example row (row index 0), before and after:** inputs InternetService='DSL', TechSupport='No' give `internet_no_techsupport` = 1.
- **Type:** binary. Built row by row, nothing is fitted, so it cannot leak across the split.

In [11]:
# 4.7 internet_no_techsupport
EX = 0
print('before (inputs):', df.loc[EX, ['InternetService', 'TechSupport']].to_dict())
df['internet_no_techsupport'] = (((df['InternetService'] != 'No') & (df['TechSupport'] == 'No'))).astype(float)
print('after: internet_no_techsupport =', df.loc[EX, 'internet_no_techsupport'])
res = F.check_feature(df.loc[is_train, 'internet_no_techsupport'], df.loc[is_train, F.TARGET])
feat['internet_no_techsupport'] = res
F.plot_feature_check('internet_no_techsupport', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_internet_no_techsupport.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'InternetService': 'DSL', 'TechSupport': 'No'}
after: internet_no_techsupport = 1.0


churn 41.8% when 1 (n=2,771) vs 11.8% when 0 (n=2,863); gap +29.9 points (approx. 95% interval +27.8 to +32.1); Spearman rho +0.34 (approx. 95% interval +0.32 to +0.36)
rule: gap >= 5 points and both groups >= 100 rows -> KEEP


**What the output shows.** On the 5,634 training customers: churn 41.8% when 1 (n=2,771) vs 11.8% when 0 (n=2,863); gap +29.9 points (approx. 95% interval +27.8 to +32.1); Spearman rho +0.34 (approx. 95% interval +0.32 to +0.36). Rule (gap >= 5 points and both groups >= 100 rows) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho +0.34 is medium (approximate 95% interval +0.32 to +0.36); the gap of +29.9 points is medium (approximate 95% interval +27.8 to +32.1). Final verdicts are set in section 5 after the redundancy check.

### 4.8 `internet_no_protection`
- **Business meaning:** Has internet and none of the four protection add-ons (security, backup, device protection, tech support).
- **Formula:** $$\text{internet\_no\_protection}=\mathbb{1}[\text{InternetService}\ne\text{No}]\cdot\mathbb{1}\Big[\sum_{k\in P}\mathbb{1}[k=\text{Yes}]=0\Big]$$
- **Exact code expression:** `((df['InternetService'] != 'No') & ((df[['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport']] == 'Yes').sum(axis=1) == 0))`
- **Input columns:** `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`
- **Why it exists (explorer finding or hypothesis):** Chart 08: internet customers with four protection add-ons leave at 5%, with none at 57%.
- **Expected direction of its link to churn:** positive
- **Example row (row index 4), before and after:** inputs InternetService='Fiber optic', OnlineSecurity='No', OnlineBackup='No', DeviceProtection='No', TechSupport='No' give `internet_no_protection` = 1.
- **Type:** binary. Built row by row, nothing is fitted, so it cannot leak across the split.

In [12]:
# 4.8 internet_no_protection
EX = 4
print('before (inputs):', df.loc[EX, ['InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']].to_dict())
df['internet_no_protection'] = (((df['InternetService'] != 'No') & ((df[['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport']] == 'Yes').sum(axis=1) == 0))).astype(float)
print('after: internet_no_protection =', df.loc[EX, 'internet_no_protection'])
res = F.check_feature(df.loc[is_train, 'internet_no_protection'], df.loc[is_train, F.TARGET])
feat['internet_no_protection'] = res
F.plot_feature_check('internet_no_protection', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_internet_no_protection.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No'}
after: internet_no_protection = 1.0


churn 56.9% when 1 (n=996) vs 20.0% when 0 (n=4,638); gap +36.9 points (approx. 95% interval +33.6 to +40.2); Spearman rho +0.32 (approx. 95% interval +0.30 to +0.34)
rule: gap >= 5 points and both groups >= 100 rows -> KEEP


**What the output shows.** On the 5,634 training customers: churn 56.9% when 1 (n=996) vs 20.0% when 0 (n=4,638); gap +36.9 points (approx. 95% interval +33.6 to +40.2); Spearman rho +0.32 (approx. 95% interval +0.30 to +0.34). Rule (gap >= 5 points and both groups >= 100 rows) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho +0.32 is medium (approximate 95% interval +0.30 to +0.34); the gap of +36.9 points is large (approximate 95% interval +33.6 to +40.2). Final verdicts are set in section 5 after the redundancy check.

### 4.9 `n_protection`
- **Business meaning:** How many of the four protection add-ons the customer has.
- **Formula:** $$\text{n\_protection}=\sum_{k\in P}\mathbb{1}[k=\text{Yes}],\ P=\{\text{security, backup, device, tech support}\}$$
- **Exact code expression:** `(df[['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport']] == 'Yes').sum(axis=1)`
- **Input columns:** `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`
- **Why it exists (explorer finding or hypothesis):** Chart 08: churn falls steadily as protection add-ons rise. It matches the explorer's `n_protection_addons`; it is rebuilt here from the raw columns so the pipeline needs no extra column.
- **Expected direction of its link to churn:** negative
- **Example row (row index 0), before and after:** inputs OnlineSecurity='No', OnlineBackup='Yes', DeviceProtection='No', TechSupport='No' give `n_protection` = 1.
- **Type:** count (0-4). Built row by row, nothing is fitted, so it cannot leak across the split.

In [13]:
# 4.9 n_protection
EX = 0
print('before (inputs):', df.loc[EX, ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']].to_dict())
df['n_protection'] = ((df[['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport']] == 'Yes').sum(axis=1)).astype(float)
print('after: n_protection =', df.loc[EX, 'n_protection'])
res = F.check_feature(df.loc[is_train, 'n_protection'], df.loc[is_train, F.TARGET])
feat['n_protection'] = res
F.plot_feature_check('n_protection', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_n_protection.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'OnlineSecurity': 'No', 'OnlineBackup': 'Yes', 'DeviceProtection': 'No', 'TechSupport': 'No'}
after: n_protection = 1.0


Spearman rho with churn -0.15 (approx. 95% interval -0.18 to -0.13); churn rate from 29.6% (lowest value/bin) to 5.5% (highest)
rule: |Spearman rho| >= 0.10 -> KEEP


**What the output shows.** On the 5,634 training customers: Spearman rho with churn -0.15 (approx. 95% interval -0.18 to -0.13); churn rate from 29.6% (lowest value/bin) to 5.5% (highest). Rule (|Spearman rho| >= 0.10) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho -0.15 is small (approximate 95% interval -0.18 to -0.13). Final verdicts are set in section 5 after the redundancy check.

### 4.10 `n_streaming`
- **Business meaning:** How many streaming add-ons (TV, movies) the customer has.
- **Formula:** $$\text{n\_streaming}=\mathbb{1}[\text{StreamingTV}=\text{Yes}]+\mathbb{1}[\text{StreamingMovies}=\text{Yes}]$$
- **Exact code expression:** `(df[['StreamingTV','StreamingMovies']] == 'Yes').sum(axis=1)`
- **Input columns:** `StreamingTV`, `StreamingMovies`
- **Why it exists (explorer finding or hypothesis):** Chart 08: streaming add-ons show a small gap (about 4 points), unlike protection add-ons. Tested here to see whether it adds anything beyond protection.
- **Expected direction of its link to churn:** weak positive
- **Example row (row index 5), before and after:** inputs StreamingTV='Yes', StreamingMovies='Yes' give `n_streaming` = 2.
- **Type:** count (0-2). Built row by row, nothing is fitted, so it cannot leak across the split.

In [14]:
# 4.10 n_streaming
EX = 5
print('before (inputs):', df.loc[EX, ['StreamingTV', 'StreamingMovies']].to_dict())
df['n_streaming'] = ((df[['StreamingTV','StreamingMovies']] == 'Yes').sum(axis=1)).astype(float)
print('after: n_streaming =', df.loc[EX, 'n_streaming'])
res = F.check_feature(df.loc[is_train, 'n_streaming'], df.loc[is_train, F.TARGET])
feat['n_streaming'] = res
F.plot_feature_check('n_streaming', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_n_streaming.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'StreamingTV': 'Yes', 'StreamingMovies': 'Yes'}
after: n_streaming = 2.0


Spearman rho with churn +0.08 (approx. 95% interval +0.06 to +0.11); churn rate from 22.6% (lowest value/bin) to 29.9% (highest)
rule: |Spearman rho| >= 0.10 -> DROP


**What the output shows.** On the 5,634 training customers: Spearman rho with churn +0.08 (approx. 95% interval +0.06 to +0.11); churn rate from 22.6% (lowest value/bin) to 29.9% (highest). Rule (|Spearman rho| >= 0.10) gives provisional verdict **DROP**. Direction matches what we expected. Scale and interval: Spearman rho +0.08 is negligible (approximate 95% interval +0.06 to +0.11). Final verdicts are set in section 5 after the redundancy check.

### 4.11 `household_size`
- **Business meaning:** Partner and dependents counted together: a proxy for household ties.
- **Formula:** $$\text{household\_size}=\mathbb{1}[\text{Partner}=\text{Yes}]+\mathbb{1}[\text{Dependents}=\text{Yes}]$$
- **Exact code expression:** `((df['Partner'] == 'Yes').astype(int) + (df['Dependents'] == 'Yes').astype(int))`
- **Input columns:** `Partner`, `Dependents`
- **Why it exists (explorer finding or hypothesis):** Ranked drivers: Dependents (V 0.16) and Partner (V 0.15) both link to lower churn. One count pools two weak signals.
- **Expected direction of its link to churn:** negative
- **Example row (row index 0), before and after:** inputs Partner='Yes', Dependents='No' give `household_size` = 1.
- **Type:** count (0-2). Built row by row, nothing is fitted, so it cannot leak across the split.

In [15]:
# 4.11 household_size
EX = 0
print('before (inputs):', df.loc[EX, ['Partner', 'Dependents']].to_dict())
df['household_size'] = (((df['Partner'] == 'Yes').astype(int) + (df['Dependents'] == 'Yes').astype(int))).astype(float)
print('after: household_size =', df.loc[EX, 'household_size'])
res = F.check_feature(df.loc[is_train, 'household_size'], df.loc[is_train, F.TARGET])
feat['household_size'] = res
F.plot_feature_check('household_size', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_household_size.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before (inputs): {'Partner': 'Yes', 'Dependents': 'No'}
after: household_size = 1.0
Spearman rho with churn -0.18 (approx. 95% interval -0.21 to -0.16); churn rate from 34.1% (lowest value/bin) to 14.0% (highest)
rule: |Spearman rho| >= 0.10 -> KEEP


**What the output shows.** On the 5,634 training customers: Spearman rho with churn -0.18 (approx. 95% interval -0.21 to -0.16); churn rate from 34.1% (lowest value/bin) to 14.0% (highest). Rule (|Spearman rho| >= 0.10) gives provisional verdict **KEEP**. Direction matches what we expected. Scale and interval: Spearman rho -0.18 is small (approximate 95% interval -0.21 to -0.16). Final verdicts are set in section 5 after the redundancy check.

### 4.12 `price_resid` (fitted feature)
- **Business meaning:** Monthly bill minus the bill the customer's service mix would normally cost. Positive means paying above the normal price for the same services.
- **Formula:** $$\text{price\_resid}=\text{MonthlyCharges}-\hat{b}_0-\sum_j \hat{b}_j\,x_j$$ where $x_j$ are the service dummies (phone, lines, internet type, six add-ons) and $\hat b$ come from a linear regression of MonthlyCharges on them, fitted on training rows only.
- **Exact code expression:** `F.ServiceAdjustedPrice().fit(train_rows).transform(rows)['price_resid']`
- **Input columns:** `MonthlyCharges` and the nine service columns.
- **Why it exists:** H6 / chart 07: leavers pay $15 more at the median, but price overlaps with fiber and add-ons. The residual isolates price from the service mix.
- **Expected direction:** positive
- **Example row:** shown in the code output below (before: bill and services; after: residual in dollars).
- **Leakage note:** this feature needs a fit, so it is rebuilt inside the model pipeline from the training rows of each fold. The column saved in `features.parquet` is fitted on the whole training split for checking and documentation only; models do not read it.

In [16]:
# 4.12 price_resid
sap = F.ServiceAdjustedPrice().fit(df.loc[is_train])
df['price_resid'] = sap.transform(df)['price_resid']
EX = int(df.loc[is_train].index[0])
print('before:', df.loc[EX, ['MonthlyCharges'] + F.SERVICE_COLS].to_dict())
print('after: price_resid =', round(df.loc[EX, 'price_resid'], 2), 'dollars above the usual price for this service mix')
res = F.check_feature(df.loc[is_train, 'price_resid'], df.loc[is_train, F.TARGET]); feat['price_resid'] = res
F.plot_feature_check('price_resid', res, base_rate, f'{PROJECT}/reports/modeling/charts/feat_price_resid.png'); plt.show()
print(res['stat']); print('rule:', res['rule'], '->', 'KEEP' if res['keep'] else 'DROP')

before: {'MonthlyCharges': 29.85, 'PhoneService': 'No', 'MultipleLines': 'No', 'InternetService': 'DSL', 'OnlineSecurity': 'No', 'OnlineBackup': 'Yes', 'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'No', 'StreamingMovies': 'No'}
after: price_resid = -0.08 dollars above the usual price for this service mix


Spearman rho with churn -0.02 (approx. 95% interval -0.05 to +0.00); churn rate from 30.0% (lowest value/bin) to 24.3% (highest)
rule: |Spearman rho| >= 0.10 -> DROP


**What the output shows.** The fitted price residual of the example row (row index 0) is -0.08 dollars: this customer pays almost exactly the usual price for the service mix. Across the training rows, Spearman rho with churn is -0.02 (negligible; approximate 95% interval -0.05 to +0.00), and churn runs from 30.0% in the lowest-residual decile to 24.3% in the highest, a drop of under 6 points with no steady trend. The rule (|rho| of at least 0.10) gives **DROP**. What this means for H6 (leavers pay about $15 more at the median): we expected a positive link, that customers paying above the normal price for their services leave more. We see none, and the sign is slightly the other way. So the higher bills of leavers are explained by what they buy (fiber, add-ons), not by paying a premium for the same services. This is a statement about this data and about association only; the data holds no price history.

## 5. Final keep-or-drop, redundancy and added value

**Drop near-duplicates, then test whether the kept features help a model at all**

1. **Why this method.** Two features that say the same thing add no information and make a linear model less stable. Separately, a feature can pass a one-at-a-time check yet add nothing once the raw columns are in the model, so we test the whole set once.
2. **Alternatives considered.**
   - Variance inflation factor (how much a feature is explained by the others): more thorough for linear models, but the simple pair correlation is enough to spot exact duplicates here.
   - Per-feature leave-one-out model runs: precise, but 12 features x 25 fits adds cost and the answer is dominated by noise at this size.
   - Keep everything and let regularisation (a penalty that shrinks weak weights) sort it out: valid for the final models, but it hides which features carry signal.
3. **How it works.** Redundancy rule, fixed before the result: in registry order, drop a feature if its absolute Spearman correlation with an earlier kept feature is 0.95 or higher. Added-value test: a regularised logistic regression (a model that adds weighted evidence from each column and turns the sum into a probability) on the raw columns versus raw plus kept features, scored by AUC (see section 3) over repeated stratified 5-fold cross-validation (the training rows are split into 5 parts; each part is scored by a model trained on the other 4; repeated 5 times with different splits) on training rows only. Rule, fixed before the result: engineered features are worth carrying if the mean AUC does not fall and the gain is larger than one SD of the paired fold-by-fold difference (the same rule as notebook 03, sections 2 and 4).
4. **How to read the output.** The correlation table shows pairs near 0.95 or more. The AUC table shows mean and standard deviation (SD, the spread across 25 fold-scores). A gain smaller than one SD is not distinguishable from noise.

In [17]:
from scipy.stats import spearmanr
names = [r['name'] for r in F.REGISTRY] + ['price_resid']
signal_keep = [n for n in names if feat[n]['keep']]
kept, dropped = [], {}
for n in names:
    if n not in signal_keep:
        dropped[n] = 'fails signal rule: ' + feat[n]['rule']; continue
    dup = next((k for k in kept if abs(spearmanr(df.loc[is_train, n], df.loc[is_train, k])[0]) >= 0.95), None)
    if dup: dropped[n] = f'duplicate of {dup} (|rho| >= 0.95)'
    else: kept.append(n)
C = pd.DataFrame({a: {b: spearmanr(df.loc[is_train, a], df.loc[is_train, b])[0] for b in names} for a in names}).round(2)
fig, ax = plt.subplots(figsize=(8, 6.5)); im = ax.imshow(C.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=60, ha='right', fontsize=8); ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
for i in range(len(names)):
    for j in range(len(names)): ax.text(j, i, f'{C.values[i,j]:.1f}', ha='center', va='center', fontsize=6)
plt.colorbar(im, label='Spearman correlation'); ax.set_title('Engineered features: rank correlation (training rows)'); plt.tight_layout()
fig.savefig(f'{PROJECT}/reports/modeling/charts/feature_correlations.png', dpi=110); plt.show()
print('KEEP :', kept); print('DROP :', dropped)

KEEP : ['contract_ord', 'm2m_fiber', 'm2m_echeck', 'm2m_first_year', 'first_year', 'log_tenure', 'internet_no_techsupport', 'internet_no_protection', 'n_protection', 'household_size']
DROP : {'n_streaming': 'fails signal rule: |Spearman rho| >= 0.10', 'price_resid': 'fails signal rule: |Spearman rho| >= 0.10'}


**Which features failed.** `n_streaming` (rho +0.08) and `price_resid` (rho -0.02) failed the signal rule (|rho| of at least 0.10), so they are dropped; the other ten passed. **Duplicates.** No pair reached the 0.95 cut-off, so no feature was dropped as a duplicate. The highest pair correlation seen is 0.94, between `m2m_first_year` and `first_year`. The heat map shows how much the features overlap (red = move together). Features that overlap strongly but are below 0.95 are kept; regularisation in the models handles that. The printed lists are the final verdicts that the models will use.

In [18]:
Xtr = df.loc[is_train].reset_index(drop=True); ytr = Xtr[F.TARGET]
splits = M.repeated_splits(ytr)
r_raw = M.cv_evaluate(M.make_pipeline('glm', feature_set='raw'), Xtr, ytr, splits)
r_eng = M.cv_evaluate(M.make_pipeline('glm', feature_set='eng', kept=kept), Xtr, ytr, splits)
cmp = pd.DataFrame({'raw columns only': r_raw[['auc','pr_auc','top20_rev','brier']].agg(['mean','std']).T.stack(),
                    'raw + kept engineered': r_eng[['auc','pr_auc','top20_rev','brier']].agg(['mean','std']).T.stack()})
display(cmp.round(4))
d = (r_eng['auc'] - r_raw['auc'])
print(f'AUC gain from engineered features: mean {d.mean():+.4f}, SD of the paired difference {d.std():.4f}, over {len(d)} fold-scores')

raw columns only  raw + kept engineered
auc       mean            0.8444                 0.8501
          std             0.0101                 0.0092
pr_auc    mean            0.6599                 0.6731
          std             0.0172                 0.0219
top20_rev mean            0.5659                 0.5543
          std             0.0210                 0.0202
brier     mean            0.1355                 0.1329
          std             0.0039                 0.0039

AUC gain from engineered features: mean +0.0057, SD of the paired difference 0.0036, over 25 fold-scores


The table shows mean and SD of each metric across 25 fold-scores for the same logistic model, with and without the engineered features. `top20_rev` is the share of leavers' monthly revenue found in the top 20% of customers by score (the business metric). `brier` is the mean squared error of the predicted probabilities (0 is perfect; lower is better). The printed gain says whether the engineered set helps beyond noise. Because the folds are paired, the difference is more precise than the two separate SDs. **Reading scale for AUC differences.** AUC is the chance that a random leaver is scored above a random stayer, so a gain of +0.01 means that in 100 random leaver-stayer pairs, one more pair is put in the right order (about 1,495 x 4,139 = 6.2 million pairs in the training rows, so +0.01 is about 62,000 pairs). As a rule of thumb here, under 0.005 is negligible, 0.005 to 0.02 small, over 0.02 meaningful; 'times the SD' compares the gain with the fold-to-fold spread of the paired difference (under 1 is indistinguishable from noise, 2 or more fairly clear). In this run the engineered set raises AUC from 0.844 to 0.850 (a gain of +0.006, about 1.6 times the SD of the paired difference, 0.004) and lowers the Brier score slightly (0.1355 to 0.1329), but the top-20% revenue capture is not better (56.6% to 55.4%, inside the spread of 2 points). So the gain is small and modest in ranking terms. If the gain is under one SD, the features are a convenience for the linear model rather than a proven improvement, and tree models are expected to find the same joint groups without help.

## 6. Write the outputs

Plain words: save the full feature table with the split column, the feature documentation table, and the list of features the models should use. Notebook 03 reads these files and never recomputes features (except in the what-if analysis, where it calls the same functions on changed inputs).

In [19]:
# consistency check: the production function gives exactly the columns built cell by cell
chk = F.add_features(pd.read_parquet(f'{PROJECT}/reports/exploration/clean.parquet'))
for n in F.ENGINEERED:
    assert np.allclose(chk[n].values, df[n].values), n
print('add_features() matches the notebook columns for all', len(F.ENGINEERED), 'row-wise features')

keep_cols = [F.ID_COL, 'split', F.TARGET] + F.RAW_COLS + F.ENGINEERED + ['price_resid']
out = df[keep_cols].copy()
out.to_parquet(f'{PROJECT}/reports/modeling/features.parquet', index=False)

example_rows = {}
for r in F.REGISTRY:
    ex = df.loc[is_train & (df[r['name']] != 0)].index[0]
    example_rows[r['name']] = ', '.join(f"{c}={df.loc[ex, c]!r}" for c in r['inputs']) + f" -> {r['name']}={df.loc[ex, r['name']]:g}"
ex = df.loc[is_train].index[0]
example_rows['price_resid'] = f"MonthlyCharges={df.loc[ex,'MonthlyCharges']}, services as listed -> price_resid={df.loc[ex,'price_resid']:.2f}"
allr = F.REGISTRY + [F.PRICE_RESID]
def cell(s): return str(s).replace('|', '/').replace('\n', ' ')
lines = ['# Feature table', '',
 'Every engineered feature, the raw columns it is built from, why it exists, and whether the models use it. Checks use the training split only (5,634 customers). Associations with churn are not causes.', '',
 'Rule for keep/drop (fixed before results): binary feature KEEP if the churn-rate gap is at least 5 points and both groups have at least 100 training rows; other features KEEP if |Spearman rho| >= 0.10; then drop a feature if it is a near-duplicate (|rho| >= 0.95) of an earlier kept one.', '',
 '| name | meaning | formula or rule | exact code expression | source columns | type | why it exists (finding or hypothesis) | expected direction | example row (before -> after) | observed on training rows | verdict |', '|' + '---|'*11]
for r in allr:
    n = r['name']; v = 'KEEP' if n in kept else 'DROP: ' + dropped[n].replace('|Spearman rho|', 'abs(Spearman rho)')
    f = r['formula'] if n == 'price_resid' else r['formula']
    lines.append('| ' + ' | '.join([f'`{n}`', cell(r['meaning']), cell('$' + f + '$'), cell('`' + r['expr'] + '`'), cell(', '.join(r['inputs'])), cell(r['kind']), cell(r['finding']), cell(r['direction']), cell(example_rows[n]), cell(feat[n]['stat']), v]) + ' |')
lines += ['', '## Raw columns used as model inputs', '',
 'Categorical (one-hot encoded inside the model pipeline): ' + ', '.join(F.RAW_CAT) + '. Numeric: ' + ', '.join(F.RAW_NUM) + '.', '',
 '## Columns deliberately left out', '', '| column | reason |', '|---|---|',
 '| customerID | identifier; kept only to join scores back |', '| Churn | text copy of the target |', '| tenure_band, monthly_charge_band | reporting bands built from tenure and MonthlyCharges |',
 '| TotalCharges, avg_monthly_spend, price_vs_history_pct | built from lifetime billing; timing relative to exit is unknown, so they may carry the outcome (see leakage check in notebook 02, section 3) |',
 '| has_internet | same information as InternetService |', '| n_addons, n_protection_addons | counts of columns already present (n_protection rebuilds the second inside the pipeline) |',
 '', '## Notes', '', '- `price_resid` is fitted (a regression on training rows). Inside the model it is refitted on the training part of every cross-validation fold. The column stored in features.parquet is fitted on the whole training split for documentation only.',
 '- `SeniorCitizen` and `gender` are raw inputs. `gender` shows no churn link in the exploration. `SeniorCitizen` does; check the policy before using an age-related field to decide who gets an offer.']
open(f'{PROJECT}/reports/modeling/features.md', 'w').write('\n'.join(lines) + '\n')
json.dump({'kept': kept, 'dropped': dropped, 'raw_cat': F.RAW_CAT, 'raw_num': F.RAW_NUM}, open(f'{PROJECT}/reports/modeling/feature_list.json', 'w'), indent=1)
print('features.parquet', out.shape, '| kept', len(kept), 'of', len(names), '| split counts', out['split'].value_counts().to_dict())

add_features() matches the notebook columns for all 11 row-wise features
features.parquet (7043, 33) | kept 10 of 12 | split counts {'train': 5634, 'test': 1409}


The output confirms the check passed, gives the shape of the saved table (7,043 rows) and how many engineered features the models will use. `features.md` carries the full table for the reader and the statistician.